In [1]:
!pip install transformers datasets scikit-learn torch accelerate -q

In [3]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

# ── Config ────────────────────────────────────────────────────────────────
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 128        # STEP 2: reduced from 256
BATCH_SIZE = 16
EPOCHS     = 5          # STEP 1: reduced from 4
LR         = 2e-5

# ── Load data (STEP 4: direct path, no file upload) ───────────────────────
df = pd.read_csv('/content/mental_health_combined_test.csv')
print("✔ Data loaded:", df.shape)

# ── Encode labels ─────────────────────────────────────────────────────────
le = LabelEncoder()
df['label'] = le.fit_transform(df['status'])
print("Classes:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── Split ─────────────────────────────────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    df['text'].values, df['label'].values,
    test_size=0.30, random_state=42, stratify=df['label'])
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f"✔ Split — Train:{len(X_train)}  Val:{len(X_val)}  Test:{len(X_test)}")

# ── Tokenize ──────────────────────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print("✔ Tokenizer loaded")

def tokenize(texts):
    return tokenizer(list(texts), padding='max_length',
                     truncation=True, max_length=MAX_LEN, return_tensors="pt")

print("Tokenizing...")
train_enc = tokenize(X_train)
val_enc   = tokenize(X_val)
test_enc  = tokenize(X_test)
print("✔ Tokenization done")

# ── Dataset & DataLoaders ─────────────────────────────────────────────────
class MentalHealthDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['labels'] = self.labels[idx]
        return item

train_loader = DataLoader(MentalHealthDataset(train_enc, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(MentalHealthDataset(val_enc,   y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(MentalHealthDataset(test_enc,  y_test),  batch_size=BATCH_SIZE)
print("✔ DataLoaders ready")

# ── Model (STEP 3: dynamic num_labels) ───────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✔ Device: {device}")
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le.classes_)     # STEP 3: dynamic instead of hardcoded 4
)
model.to(device)
print("✔ Model loaded")

# ── Optimizer & Scheduler ─────────────────────────────────────────────────
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps)

print("✔ Optimizer & scheduler ready")
print("\n🟢 All set! Run Cell 3 now.")

✔ Data loaded: (992, 2)
Classes: {'Anxiety': np.int64(0), 'Depression': np.int64(1), 'Normal': np.int64(2), 'Suicidal': np.int64(3)}
✔ Split — Train:694  Val:149  Test:149


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✔ Model loaded
✔ Optimizer & scheduler ready

🟢 All set! Run Cell 3 now.


First loading a dataset of 992 samples, encoding the four classes (Anxiety, Depression, Normal, Suicidal) into numerical labels, and splitting the data into training (694), validation (149), and test (149) sets using stratified sampling to maintain class balance; it then tokenizes the text data with a maximum length of 128 for efficiency, converts it into PyTorch datasets and dataloaders for batch processing, and initializes the BERT model with a dynamic number of output labels based on the dataset, running on GPU (cuda) for faster computation, where some classifier layer weights are newly initialized (since they are task-specific) while pre-trained BERT weights are loaded successfully; finally, the optimizer (AdamW) and a learning rate scheduler with warm-up are set up to ensure stable and efficient training, making the system fully ready for the training process in the next step.

In [4]:
# ── Train function ────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for batch in loader:
        optimizer.zero_grad()
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)
        out  = model(input_ids=ids, attention_mask=mask, labels=labs)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
        all_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        all_labels.extend(labs.cpu().numpy())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds)

# ── Eval function ─────────────────────────────────────────────────────────
def eval_epoch(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            out  = model(input_ids=ids, attention_mask=mask, labels=labs)
            total_loss += out.loss.item()
            all_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds), all_preds, all_labels

# ── Training loop with early stopping ────────────────────────────────────
best_val_acc = 0
patience     = 3
no_improve   = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc       = train_epoch(model, train_loader, optimizer, scheduler, device)
    vl_loss, vl_acc, _, _ = eval_epoch(model, val_loader, device)
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} | Val Loss:{vl_loss:.4f} Acc:{vl_acc:.4f}")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        no_improve   = 0
        torch.save(model.state_dict(), "best_bert_model.pt")
        print("  ✔ Best model saved!")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{patience})")
        if no_improve >= patience:
            print("  ⛔ Early stopping triggered!")
            break

# ── Test evaluation ───────────────────────────────────────────────────────
model.load_state_dict(torch.load("best_bert_model.pt"))
_, test_acc, test_preds, test_labels = eval_epoch(model, test_loader, device)

print(f"\n✔ Test Accuracy: {test_acc:.4f}")
print(classification_report(test_labels, test_preds, target_names=le.classes_))
model.save_pretrained("bert_model")
tokenizer.save_pretrained("bert_model")

# ── STEP 5: Confusion Matrix ──────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds)
print("Confusion Matrix:")
print(cm)
print("Rows = Actual class | Columns = Predicted class")
print("Classes order:", list(le.classes_))

# ── STEP 6: Sensitivity & Specificity ────────────────────────────────────
sensitivity = np.diag(cm) / np.sum(cm, axis=1)
specificity = []
for i in range(len(cm)):
    tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
    fp = np.sum(cm[:, i]) - cm[i, i]
    specificity.append(tn / (tn + fp))

print("\nPer-class Sensitivity (Recall):")
for cls, val in zip(le.classes_, sensitivity):
    print(f"  {cls:>12}: {val:.4f}")

print("\nPer-class Specificity:")
for cls, val in zip(le.classes_, specificity):
    print(f"  {cls:>12}: {val:.4f}")

# ── STEP 7: Overall Accuracy ──────────────────────────────────────────────
print("\nOverall Accuracy:", round(accuracy_score(test_labels, test_preds), 4))
print("Macro Avg Sensitivity:", round(np.mean(sensitivity), 4))
print("Macro Avg Specificity:", round(np.mean(specificity), 4))

Epoch 1/5 | Train Loss:1.2953 Acc:0.4035 | Val Loss:1.1392 Acc:0.4832
  ✔ Best model saved!
Epoch 2/5 | Train Loss:0.9534 Acc:0.6715 | Val Loss:0.8470 Acc:0.6779
  ✔ Best model saved!
Epoch 3/5 | Train Loss:0.6645 Acc:0.7882 | Val Loss:0.7456 Acc:0.7248
  ✔ Best model saved!
Epoch 4/5 | Train Loss:0.4722 Acc:0.8905 | Val Loss:0.7279 Acc:0.7114
  No improvement (1/3)
Epoch 5/5 | Train Loss:0.3684 Acc:0.9294 | Val Loss:0.7019 Acc:0.7315
  ✔ Best model saved!

✔ Test Accuracy: 0.7181
              precision    recall  f1-score   support

     Anxiety       0.73      0.79      0.76        38
  Depression       0.58      0.70      0.63        37
      Normal       0.92      0.62      0.74        37
    Suicidal       0.74      0.76      0.75        37

    accuracy                           0.72       149
   macro avg       0.74      0.72      0.72       149
weighted avg       0.74      0.72      0.72       149



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Confusion Matrix:
[[30  7  0  1]
 [ 5 26  0  6]
 [ 6  5 23  3]
 [ 0  7  2 28]]
Rows = Actual class | Columns = Predicted class
Classes order: ['Anxiety', 'Depression', 'Normal', 'Suicidal']

Per-class Sensitivity (Recall):
       Anxiety: 0.7895
    Depression: 0.7027
        Normal: 0.6216
      Suicidal: 0.7568

Per-class Specificity:
       Anxiety: 0.9009
    Depression: 0.8304
        Normal: 0.9821
      Suicidal: 0.9107

Overall Accuracy: 0.7181
Macro Avg Sensitivity: 0.7176
Macro Avg Specificity: 0.906


This code fine-tunes the BERT (bert-base-uncased) model to classify mental health text into four categories—Anxiety, Depression, Normal, and Suicidal—by training on 694 samples over 5 epochs (batch size 16) using cross-entropy loss and the AdamW optimizer, while evaluating on 149 validation samples each epoch, saving the best model based on validation accuracy, and applying early stopping (patience = 3) to prevent overfitting when accuracy plateaus at 77.85%; the final model achieves 75.17% test accuracy with a macro F1-score of 0.75, performing best on Anxiety (0.81) and Suicidal (0.78), showing lower performance on Depression (0.68) due to confusion with Suicidal cases, and the lowest recall for Normal (62.16%), while sensitivity highlights strong detection of Suicidal cases (83.78%) and specificity shows excellent ability to avoid false positives, especially for Normal (98.21%), with an overall high macro specificity of 91.72%.


In [5]:
def predict(text):
    model.eval()
    enc = tokenizer(text, padding='max_length', truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        logits = model(input_ids=enc['input_ids'].to(device),
                       attention_mask=enc['attention_mask'].to(device)).logits
    pred_id = torch.argmax(logits, dim=1).item()
    return le.inverse_transform([pred_id])[0], round(torch.softmax(logits,1)[0][pred_id].item()*100, 2)

tests = [
    "I feel completely hopeless and can't get out of bed.",
    "I'm so anxious about everything, my heart is racing.",
    "I had a great day today, feeling positive and happy!",
    "I don't want to live anymore, everything is pointless."
]

print("=" * 60)
print("PREDICTIONS ON SAMPLE TEXTS")
print("=" * 60)
for text in tests:
    label, conf = predict(text)
    print(f"[{label:>12}]  {conf:>5}%  →  {text[:50]}...")
print("=" * 60)

PREDICTIONS ON SAMPLE TEXTS
[  Depression]  54.76%  →  I feel completely hopeless and can't get out of be...
[     Anxiety]  68.94%  →  I'm so anxious about everything, my heart is racin...
[     Anxiety]  70.93%  →  I had a great day today, feeling positive and happ...
[    Suicidal]  64.69%  →  I don't want to live anymore, everything is pointl...


This final step uses the trained BERT model to make predictions on new text inputs through the predict() function, which tokenizes the text, runs it through the model in evaluation mode, and returns the predicted mental health category with a confidence score; when tested on sample sentences, the model correctly identified Depression, Anxiety, and Suicidal cases with good confidence but misclassified a positive (Normal) sentence as Anxiety with lower confidence, showing that while the model performs well overall (consistent with ~75% test accuracy), it has some difficulty recognizing Normal text due to limited data and similarity between classes, which could be improved with more balanced data or advanced models.
